<a href="https://colab.research.google.com/github/meryllinoliveira/Projetos/blob/main/Projetos/Desafio1_API_Python_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Instala as bibliotecas usadas no guia.
!pip install -q gspread faker


In [8]:
# Importa as ferramentas para autenticação, geração de dados e sorteios.
import json, gspread
from google.colab import userdata
from faker import Faker
import random

# Lê a credencial salva nos Secrets e conecta à planilha.
info = json.loads(userdata.get("GCP_SA_JSON"))
gc = gspread.service_account_from_dict(info)
# Exibe o e-mail da conta autenticada.
print("Conectado como:", info["client_email"])

Conectado como: desafio1-python@unified-destiny-513817-p9.iam.gserviceaccount.com


In [11]:
# Informe o ID da planilha e o nome que será dado à aba.
SHEET_ID = "1NhOZRkg3TEBBX_uor2F73P0XmN3Y9m1_7VLtCigZeWY"
NOME_TABELA = "Streaming de filmes"   # nome da tabela na ficha do tema, ex.: "vagas"

# Abre a planilha e seleciona sua primeira aba.
sh = gc.open_by_key(SHEET_ID)
ws = sh.sheet1
# Renomeia a aba e exibe os dados da planilha.
ws.update_title(NOME_TABELA)     # a aba passa a ter o nome da tabela
print(sh.title, "|", ws.title, "|", sh.url)

Desafio_01_Filmes_em_Streaming | Streaming de filmes | https://docs.google.com/spreadsheets/d/1NhOZRkg3TEBBX_uor2F73P0XmN3Y9m1_7VLtCigZeWY


In [13]:
# Substitua pelos nomes EXATOS de coluna da ficha do seu tema
# Define as colunas que formarão o cabeçalho da tabela.
COLUNAS = ["filmes_id", "titulo", "genero", "ano_lançamento", "duracao_min", "classificacao_indicativa", "diretor_ficticio"]

# Escreve o cabeçalho na primeira célula e confere o resultado.
ws.update(range_name="A1", values=[COLUNAS])
print("Cabeçalho confere:", ws.row_values(1) == COLUNAS)

Cabeçalho confere: True


In [18]:
# Configura os geradores aleatórios para produzir resultados reproduzíveis.
fake = Faker("pt_BR")
Faker.seed(42)
random.seed(42)

def gerar_linha(i):
    # Cria um registro fictício para as colunas definidas no esquema.
    # Personalize de acordo com o seu esquema de dados
    return {
        "filmes_id": i,
        "titulo":f"{fake.word().capitalize()} {fake.word().capitalize()}",
        "genero":random.choices(
            ["ação","comédia", "drama", "terror",  "documentário", "animação", "ficção científica", "romance"],
            weights = [18, 16, 15, 12, 11, 10, 10, 8],
            k=1
        )[0],
        "ano_lançamento": random.randint(1985, 2024),
        "duracao_min": random.randint(75, 184),
        "classificacao_indicativa": random.choice(["L", "10", "12", "14", "16", "18"]),
        "diretor_ficticio": fake.name()
    }

# Gera 10 registros numerados de 1 a 10.
linhas = [gerar_linha(i) for i in range(1, 11)]

# Organiza os valores na ordem do cabeçalho; nome de coluna incorreto gera KeyError.
valores = [[linha[c] for c in COLUNAS] for linha in linhas]
# Acrescenta os registros à planilha e informa quantos foram enviados.
ws.append_rows(valores, value_input_option="RAW")
print(len(valores), "linhas enviadas com sucesso")

10 linhas enviadas com sucesso
